# Predictive Modeling

In [9]:
# Importing my new CSV as my model data

import pandas as pd

crfedf = pd.read_csv("../data/processed/crfedf.csv")

In [10]:
crfedf.head()

,player_name,position,class_year,previous_school,school_name,division,conference_code,primary_conference,source_url,collected_at,height_inches,hometown_city,hometown_state,hometown_country,season,height_imputed
0,Becca Siedenburg,S,SR,Gardner-Webb,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,68.0,Wales,WI,USA,2025,False
1,Avery Thaler,MB,SO,NaN,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,70.0,Fairfield,TX,USA,2025,False
2,Rachel Koss,S,JR,NaN,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,71.0,Freedom,WI,USA,2025,False
3,Courtney Church,OH,R-FR,NaN,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,72.0,Driftwood,TX,USA,2025,False
4,Hannah Gonzalez,MB,JR,NaN,Abilene Christian University,DI,WAC,Western Athletic Conference,https://acusports.com/sports/womens-volleyball...,2026-07-20T00:46:56.672717+00:00,74.0,Lucas,TX,USA,2025,False


## Inserting Packages

In [11]:
import numpy as np
import pandas as pd

In [12]:
# Making the class years a numerical value for totals
class_map = {
    "FR": 1,
    "R-FR": 2,
    "SO": 2,
    "R-SO": 3,
    "JR": 3,
    "R-JR": 4,
    "SR": 4,
    "R-SR": 5,
    "GR": 5,
    "GRAD": 5,
    "GRADUATE": 5
}

crfedf["experience"] = (
    crfedf["class_year"]
    .astype(str)
    .str.upper()
    .str.strip()
    .map(class_map)
)

In [13]:
# Indicating number of transfers
crfedf["is_transfer"] = crfedf["previous_school"].notna()

In [14]:
# Indicating if the player is international
crfedf["is_international"] = (
    crfedf["hometown_country"]
    .fillna("USA")
    .str.upper()
    != "USA"
)

In [15]:
# State Lookup to convert to state codes
state_lookup = {
    "Alabama": "AL",
    "Alaska": "AK",
    "Arizona": "AZ",
    # ...
}

## Creating "teamdf"

In [16]:
teamdf = (
    crfedf
    .groupby(
        ["school_name", "season"],
        as_index=False
    )
)

In [17]:
# Aggregating the teamdf
teamdf = (
    crfedf
    .groupby(
        ["school_name", "season"],
        as_index=False
    )
    .agg(
        roster_size=("player_name", "count"),

        avg_height=("height_inches", "mean"),
        std_height=("height_inches", "std"),

        avg_experience=("experience", "mean"),

        transfers=("is_transfer", "sum"),

        international_players=("is_international", "sum"),

        conference=("conference_code", "first"),

        division=("division", "first")
    )
)